# Learn-a-World v2: Full Training & Promotion Campaign
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sahiltolani30/learn-a-world/blob/v2/notebooks/02_train_v2_colab.ipynb)

**A JEPA-style latent world model with learned reward-value planning, ~300k starting dataset, multi-step unroll consistency, and locked promotional evaluation.**

### Campaign Workflow:
1. **Drive Persistence**: Mounts Google Drive so all ~300k transitions, checkpoints, and scorecards survive session disconnects.
2. **Stage 1 (Dataset Collection)**: Collects 300,000 sequence transitions (150,000 `structured_random` + 150,000 `teacher_hybrid`).
3. **Stage 2 (Task Heads Training)**: Trains `EventHead` (Focal BCE), `HealthHead` (Huber), `QEnsemble` (TwoHot 51 bins), and `PolicyPrior` (BC) for 5,000 steps on GPU.
4. **Stage 3 (Phase 3 Joint Rounds)**: 3-round collect-retrain loop with multi-step imagination unrolls and dynamic consistency.
5. **Stage 4 (Phase 4 Locked Comparison)**: 100-episode evaluation on fixed seeds (1000–1099) across 4 agents (`v1_hybrid`, `v1_pure`, `v2_pure`, `random`) with automatic promotion decision rule.


### Step 1: Mount Google Drive & Environment Setup
Mounts Google Drive to persist all shards, model checkpoints, and results permanently.

In [1]:
from google.colab import drive
import os
from pathlib import Path

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create persistent workspace directories in Drive
DRIVE_DIR = Path('/content/drive/MyDrive/learn-a-world')
(DRIVE_DIR / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'data' / 'sequences').mkdir(parents=True, exist_ok=True)
(DRIVE_DIR / 'results').mkdir(parents=True, exist_ok=True)

print('Google Drive persistent directory initialized at:', DRIVE_DIR)


Mounted at /content/drive
Google Drive persistent directory initialized at: /content/drive/MyDrive/learn-a-world


### Step 2: Clone or Pull Repository on Branch `v2`

In [2]:
%%bash
cd /content
if [ ! -d "learn-a-world" ]; then
  git clone https://github.com/sahiltolani30/learn-a-world.git
fi
cd learn-a-world
git fetch origin
git checkout v2
git pull origin v2
echo "--> Active branch and latest commit:"
git log -1 --oneline


branch 'v2' set up to track 'origin/v2'.
Already up to date.
--> Active branch and latest commit:
f62bac8 Created using Colab


Cloning into 'learn-a-world'...
Switched to a new branch 'v2'
From https://github.com/sahiltolani30/learn-a-world
 * branch            v2         -> FETCH_HEAD


### Step 3: Install Dependencies & Verify GPU

In [3]:
%%bash
cd /content/learn-a-world
pip install -q -r requirements.txt
pip install -q crafter gymnasium
nvidia-smi


Tue Oct 27 00:30:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 4: Verify Base Checkpoint & Run Pre-Flight Tests
Ensures the base v1 checkpoint (`checkpoints/jepa_crafter.pt`) is present and passes the complete 43-test test suite.

In [ ]:
import os, shutil
from pathlib import Path

# Verify / sync base v1 checkpoint
src_ckpt = Path('/content/learn-a-world/checkpoints/jepa_crafter.pt')
dst_ckpt = Path('/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter.pt')

if src_ckpt.exists() and not dst_ckpt.exists():
    shutil.copyfile(src_ckpt, dst_ckpt)
    print(f'--> Copied base v1 checkpoint to Drive: {dst_ckpt}')
elif dst_ckpt.exists() and not src_ckpt.exists():
    src_ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(dst_ckpt, src_ckpt)
    print(f'--> Copied checkpoint from Drive to local repo: {src_ckpt}')
elif src_ckpt.exists() and dst_ckpt.exists():
    print(f'--> Base checkpoint ready in both local repo and Drive: {dst_ckpt}')
else:
    print('--> Notice: Checkpoint will be fetched via git or downloaded during campaign.')

# Run pre-flight verification test suite (43 tests)
!cd /content/learn-a-world && pytest tests/ -v


### Step 5: Option A — Launch Full One-Click Campaign (~300k Steps)
Executes all 4 stages automatically:
- **Stage 1**: 300k Dataset (150k structured random exploration + 150k teacher hybrid play)
- **Stage 2**: 5,000 Head Training Steps (EventHead + HealthHead + QEnsemble + PolicyPrior)
- **Stage 3**: 3-Round Joint Fine-Tuning & Retrain Loop (`run_step3_rounds`)
- **Stage 4**: 100-Episode Locked Final Comparison (seeds 1000..1099) & Promotion Decision

In [ ]:
%%bash
cd /content/learn-a-world
python training/run_v2_colab_campaign.py \
    --drive-dir /content/drive/MyDrive/learn-a-world \
    --n-transitions 150000 \
    --head-steps 5000 \
    --rounds 3 \
    --round-transitions 30000 \
    --joint-steps 1000 \
    --eval-episodes 100


### Step 6: Option B — Run Staged (Modular Execution)
If you prefer running stage by stage to inspect intermediate progress, use the separate cells below:

In [ ]:
# 6A: Collect 300k starting transitions (150k structured_random + 150k teacher_hybrid)
!cd /content/learn-a-world && python training/run_v2_colab_campaign.py \
    --drive-dir /content/drive/MyDrive/learn-a-world \
    --stage collect \
    --n-transitions 150000


In [ ]:
# 6B: Train Task Heads for 5,000 steps
!cd /content/learn-a-world && python training/run_v2_colab_campaign.py \
    --drive-dir /content/drive/MyDrive/learn-a-world \
    --stage heads \
    --head-steps 5000


In [ ]:
# 6C: Phase 3 Joint Fine-Tuning & 3-Round Collect-Retrain Loop
!cd /content/learn-a-world && python training/run_v2_colab_campaign.py \
    --drive-dir /content/drive/MyDrive/learn-a-world \
    --stage rounds \
    --rounds 3 \
    --round-transitions 30000 \
    --joint-steps 1000


In [ ]:
# 6D: Phase 4 Locked 100-Episode Final Comparison & Promotion Decision
!cd /content/learn-a-world && python training/run_v2_colab_campaign.py \
    --drive-dir /content/drive/MyDrive/learn-a-world \
    --stage final \
    --eval-episodes 100


### Step 7: Inspect Final Promotion Report & Decision Scorecard
Displays the comparison table across all 4 agents, promotion criteria audit, and per-achievement unlock rates.

In [ ]:
import os
from IPython.display import Markdown, display

report_file = '/content/drive/MyDrive/learn-a-world/results/promotion_decision.md'
if not os.path.exists(report_file):
    report_file = '/content/learn-a-world/results/promotion_decision.md'

if os.path.exists(report_file):
    display(Markdown(open(report_file).read()))
else:
    print('Report not found yet. Run Step 5 or 6D first.')


In [ ]:
# 1. Mount Google Drive to access your checkpoint
from google.colab import drive
import os

drive.mount('/content/drive')

# 2. Navigate to repository and pull latest commit (15e8ad9)
%cd /content/learn-a-world
!git pull origin main

# 3. Verify checkpoint exists
ckpt_path = "/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt"
print(f"Checkpoint exists: {os.path.exists(ckpt_path)}")

# 4. Run 50-Episode Screening with Pure Cognitive Causal Architecture (Tabula Rasa)
!python scripts/screen_v3_dual_memory.py \
  --checkpoint "/content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt" \
  --memory_bank "checkpoints/tabula_rasa_memory_bank.pt" \
  --drive_dir "/content/drive/MyDrive/learn-a-world" \
  --num_episodes 50 \
  --seed_start 2050 \
  --max_steps 1000 \
  --horizon 10 \
  --num_candidates 64 \
  --cem_iterations 2 \
  --prior_mix 0.5 \
  --uncertainty_weight 0.5 \
  --novelty_weight 0.02 \
  --device cuda \
  --output_json "results/screen_v3_cognitive_causal_50ep.json" \
  --empty_memory \
  --save_updated_memory


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/learn-a-world
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 660 bytes | 220.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   56e8e1e..15e8ad9  main       -> origin/main
Updating 56e8e1e..15e8ad9
Fast-forward
 scripts/screen_v3_50eps.py       | 10 ++++++++--
 scripts/screen_v3_dual_memory.py | 10 ++++++++--
 2 files changed, 16 insertions(+), 4 deletions(-)
Checkpoint exists: True
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and reque

In [ ]:
import torch
import numpy as np
from planning.semantic_memory import CRAFTER_INVENTORY_KEYS, vector_to_inventory_dict

# Load the memory bank saved by the agent
bank_path = "checkpoints/tabula_rasa_memory_bank.pt"
data = torch.load(bank_path, map_location="cpu")

total_memories = len(data["keys"])
actions = data["actions"].numpy()
returns = data["returns"].numpy()
names = data["outcome_names"]
inventories = data["inventories"].numpy()
deltas = data["inventory_deltas"].numpy()

print("=" * 80)
print(f"       AGENT'S EXPERIENCE DIARY: {total_memories} TOTAL LEARNED MEMORIES")
print("=" * 80)

# 1. Print Breakdown by Discovery Type
discovery_counts = {}
for name in names:
    base_name = name.split(",")[0].strip()
    discovery_counts[base_name] = discovery_counts.get(base_name, 0) + 1

print("\n--- Summary of What the Agent Learned ---")
for event, count in sorted(discovery_counts.items(), key=lambda x: -x[1]):
    print(f"  {event:<30}: {count} memories")

# 2. Inspect Sample Breakthrough Memories (What Worked!)
print("\n--- Sample Breakthrough Discoveries (High Return G > 1.0) ---")
high_return_indices = np.where(returns >= 1.0)[0]
for idx in high_return_indices[:10]:
    inv = vector_to_inventory_dict(inventories[idx])
    delta = vector_to_inventory_dict(deltas[idx])
    delta_str = ", ".join([f"{k}: {'+' if v > 0 else ''}{v}" for k, v in delta.items() if v != 0])
    print(f"  [Memory #{idx:03d}] Event: {names[idx]:<20} | Act: {actions[idx]:<2d} | Wood Held: {inv.get('wood', 0)} | Changed: {delta_str or 'None':<25} | Return G: {returns[idx]:.2f}")

# 3. Inspect Sample Mistake Memories (What Failed / G = 0.0)
print("\n--- Sample Learned Mistakes (Suppressed Fruitless Actions G = 0.0) ---")
fruitless_indices = np.where(returns == 0.0)[0]
for idx in fruitless_indices[:5]:
    inv = vector_to_inventory_dict(inventories[idx])
    print(f"  [Mistake #{idx:03d}] Action Attempted: {actions[idx]:<2d} | Backpack Held: wood={inv.get('wood', 0)} | Result: G = 0.0 (Suppressed from repeating)")
print("=" * 80)



import torch
import numpy as np
from planning.semantic_memory import CRAFTER_INVENTORY_KEYS, vector_to_inventory_dict

data = torch.load("checkpoints/tabula_rasa_memory_bank.pt", map_location="cpu")
actions = data["actions"].numpy()
returns = data["returns"].numpy()
names = data["outcome_names"]
inventories = data["inventories"].numpy()
deltas = data["inventory_deltas"].numpy()

print("=" * 80)
print("     THE AGENT'S PURE TOOL & MILESTONE CAUSAL DISCOVERIES")
print("=" * 80)

# Filter out passive hunger/thirst ticks
milestone_keywords = ["wood", "table", "pickaxe", "sword", "cow", "zombie", "skeleton", "fruitless"]

for idx, name in enumerate(names):
    if any(k in name.lower() for k in milestone_keywords):
        inv = vector_to_inventory_dict(inventories[idx])
        delta = vector_to_inventory_dict(deltas[idx])
        delta_str = ", ".join([f"{k}: {'+' if v > 0 else ''}{v}" for k, v in delta.items() if v != 0])
        status = "BREAKTHROUGH" if returns[idx] > 0 else "MISTAKE LEARNED"
        print(f"[{status:<15}] #{idx:03d} | {name:<22} | Act: {actions[idx]:<2d} | Wood Held: {inv.get('wood', 0)} | Delta: {delta_str or 'None':<22} | G: {returns[idx]:.2f}")



       AGENT'S EXPERIENCE DIARY: 901 TOTAL LEARNED MEMORIES

--- Summary of What the Agent Learned ---
  gain(drink-1)                 : 187 memories
  gain(food-1)                  : 148 memories
  gain(energy-1)                : 94 memories
  gain(energy+1)                : 47 memories
  gain(health+1)                : 42 memories
  collect_wood                  : 35 memories
  collect_sapling               : 34 memories
  place_plant                   : 32 memories
  wake_up                       : 27 memories
  gain(health-2)                : 25 memories
  gain(sapling+1)               : 23 memories
  fruitless_act_15              : 22 memories
  place_table                   : 19 memories
  gain(wood+1)                  : 19 memories
  fruitless_act_10              : 19 memories
  fruitless_act_11              : 17 memories
  gain(sapling-1)               : 16 memories
  collect_drink                 : 9 memories
  fruitless_act_14              : 9 memories
  fruitless_act_16     

     THE AGENT'S PURE TOOL & MILESTONE CAUSAL DISCOVERIES
[BREAKTHROUGH   ] #003 | collect_wood           | Act: 5  | Wood Held: 0 | Delta: wood: +1               | G: 5.06
[BREAKTHROUGH   ] #005 | place_table            | Act: 8  | Wood Held: 2 | Delta: wood: -2               | G: 2.45
[BREAKTHROUGH   ] #018 | gain(wood+1)           | Act: 5  | Wood Held: 1 | Delta: wood: +1               | G: 3.53
[MISTAKE LEARNED] #024 | fruitless_act_10       | Act: 10 | Wood Held: 0 | Delta: None                   | G: 0.00
[MISTAKE LEARNED] #025 | fruitless_act_15       | Act: 15 | Wood Held: 0 | Delta: None                   | G: 0.00
[MISTAKE LEARNED] #026 | fruitless_act_15       | Act: 15 | Wood Held: 0 | Delta: None                   | G: 0.00
[BREAKTHROUGH   ] #032 | collect_wood           | Act: 5  | Wood Held: 0 | Delta: wood: +1, health: +1   | G: 1.32
[BREAKTHROUGH   ] #033 | place_table            | Act: 8  | Wood Held: 2 | Delta: wood: -2               | G: 1.00
[BREAKTHROUGH   ] #052

In [ ]:
%cd /content/learn-a-world
!git pull origin main
!python scripts/inspect_memory_bank.py --bank /content/drive/MyDrive/learn-a-world/checkpoints/tabula_rasa_memory_bank.pt


/content/learn-a-world
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.41 KiB | 282.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   15e8ad9..5c95660  main       -> origin/main
Updating 15e8ad9..5c95660
Fast-forward
 planning/semantic_memory.py    |  17 +++-
 scripts/inspect_memory_bank.py | 214 +++++++++++++++++++++++++++++++++++++++++
 2 files changed, 227 insertions(+), 4 deletions(-)
 create mode 100644 scripts/inspect_memory_bank.py
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.or

In [ ]:
%cd /content/learn-a-world
!git pull origin main

!python scripts/inspect_memory_bank.py \
    --bank /content/drive/MyDrive/learn-a-world/checkpoints/tabula_rasa_memory_bank.pt \
    --audit
%cd /content/learn-a-world
!git pull origin main

!python scripts/dump_complete_memory.py \
    --bank /content/drive/MyDrive/learn-a-world/checkpoints/tabula_rasa_memory_bank.pt \
    --save_clean


/content/learn-a-world
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 3.85 KiB | 1.92 MiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   825fc09..6ceb3af  main       -> origin/main
Updating 825fc09..6ceb3af
Fast-forward
 scripts/dump_complete_memory.py | 251 ++++++++++++++++++++++++++++++++++++++++
 1 file changed, 251 insertions(+)
 create mode 100644 scripts/dump_complete_memory.py
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I000

In [ ]:
%cd /content/learn-a-world
!git pull origin main

!python scripts/screen_v3_dual_memory.py \
    --num_episodes 50 \
    --seed_start 2100 \
    --memory_bank /content/drive/MyDrive/learn-a-world/checkpoints/clean_optimal_memory_bank.pt


/content/learn-a-world
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11 (delta 8), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 9.38 KiB | 480.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   6ceb3af..3d14aee  main       -> origin/main
Updating 6ceb3af..3d14aee
Fast-forward
 planning/mpc_v3.py            |  50 +++++++++++++--
 planning/semantic_memory.py   | 144 +++++++++++++++++++++++++++++++++++++++++-
 tests/test_semantic_memory.py |  82 ++++++++++++++++++++++++
 3 files changed, 269 insertions(+), 7 deletions(-)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at htt

In [ ]:
%cd /content/learn-a-world
!git pull origin main

!python3 scripts/screen_v3_dual_memory.py \
  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --memory_bank /content/drive/MyDrive/learn-a-world/checkpoints/clean_optimal_memory_bank.pt \
  --episodes 50 \
  --start_seed 2100 \
  --output_json results/screen_v3_tether_fixed_50ep.json


/content/learn-a-world
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 7 (delta 5), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 9.66 KiB | 1.93 MiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   3d14aee..3c91998  main       -> origin/main
Updating 3d14aee..3c91998
Fast-forward
 planning/mpc_v3.py            |  90 +++++++++++++++++++++++++++++--
 planning/semantic_memory.py   | 122 +++++++++++++++++++++++++++++++++++++++---
 tests/test_semantic_memory.py |  69 ++++++++++++++++++++++++
 3 files changed, 271 insertions(+), 10 deletions(-)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration 

In [ ]:
%cd /content/learn-a-world

# 1. Fetch latest changes and cleanly reset to origin/main
!git fetch origin
!git reset --hard origin/main

# 2. Verify the file exists and shows the latest commit
!ls -l scripts/run_phase0_chain_diagnostics.py
!git log -n 1 --oneline


/content/learn-a-world
HEAD is now at 1788ef4 feat(causal): implement Phase 0 diagnostics and Phase 1 transition learning
-rw-r--r-- 1 root root 1505 Oct 26 23:07 scripts/run_phase0_chain_diagnostics.py
1788ef4 (HEAD -> v2, origin/main, origin/HEAD) feat(causal): implement Phase 0 diagnostics and Phase 1 transition learning


In [ ]:
# Cell 2: Run Phase 0 Diagnostics
!python3 scripts/run_phase0_chain_diagnostics.py \
  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --memory_bank /content/drive/MyDrive/learn-a-world/checkpoints/clean_optimal_memory_bank.pt \
  --episodes 3 \
  --max_steps 300 \
  --seed_start 2050 \
  --device cuda \
  --output_json results/phase0_chain_diagnostics.json


Streaming output truncated to the last 5000 lines.
            {
              "step": 68,
              "action": 0,
              "reward": 0.0,
              "done": false,
              "planner": {
                "reward_sum": 0.007585172075778246,
                "terminal_value": 0.5073722004890442,
                "uncertainty_penalty": 0.009583336301147938,
                "tte_gain": 0.0,
                "best_score": 0.7024330496788025,
                "mean_q_std": 0.0061169397085905075,
                "memory_retrieved_action": -1.0,
                "memory_match_sim": 0.0,
                "memory_best_q": 0.0,
                "active_subgoal_status": "",
                "active_subgoal_target": "",
                "structure_offset_x": 0.0,
                "structure_offset_y": 0.0,
                "stone_offset_x": 0.0,
                "stone_offset_y": 0.0
              },
              "predicted_transition": {
                "state_delta": [
                  -0.20

In [ ]:
# Cell 3: Data Collection with State Deltas
!python3 training/collector_v3.py \
  --model_checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --drive_dir /content/drive/MyDrive/learn-a-world/data/phase1_transitions \
  --target_transitions 2000 \
  --seed_start 5000 \
  --tranche_id 1 \
  --device cuda


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00:1793056213.194451   44015 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecatio

In [ ]:
# Cell 4: Train Phase 1 Transition Model
!python3 training/run_phase1_transition_learning.py \
  --shards_dir /content/drive/MyDrive/learn-a-world/data/phase1_transitions \
  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --output_checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_phase1_transitions.pt \
  --output_json results/phase1_transition_training.json \
  --steps 1000 \
  --seq_len 16 \
  --horizons 1 5 10 \
  --device cuda


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00:1793056463.769244   45053 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecatio

In [ ]:
# Cell 5: Inspect Phase 1 Verification Metrics
import json
with open("results/phase1_transition_training.json") as f:
    res = json.load(f)

print("Status:", res.get("status"))
print("Dataset transitions:", res.get("dataset", {}).get("total_transitions"))
print("Metrics:")
print(json.dumps(res.get("metrics"), indent=2))


Status: completed
Dataset transitions: 1985
Metrics:
{
  "state_delta_mse": 0.014680476568173617,
  "state_delta_persistence_mse": 0.016876220703125,
  "state_delta_valid_transitions": 4096.0,
  "reward_mae": 0.35379588324576616,
  "termination_accuracy": 0.999755859375,
  "counterfactual_action_separation": 0.00048047341260826215
}


In [ ]:
import json
import shutil
from pathlib import Path
from google.colab import files

# 1. Paths
json_path = Path("/content/learn-a-world/results/phase0_chain_diagnostics.json")
drive_dest = Path("/content/drive/MyDrive/learn-a-world/results")

# 2. Backup to Google Drive if connected
if drive_dest.parent.exists():
    drive_dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(json_path, drive_dest / "phase0_chain_diagnostics.json")
    print(f" Saved to Drive: {drive_dest / 'phase0_chain_diagnostics.json'}")

# 3. Print Clean, Compact Summary (No Truncation!)
with open(json_path, "r") as f:
    data = json.load(f)

print("\n" + "=" * 80)
print(f" PHASE 0 CHAIN DIAGNOSTICS SUMMARY ({data.get('num_episodes')} eps, Seeds: {data.get('seed_range')})")
print("=" * 80)

for res in data.get("results", []):
    name = res.get("ablation")
    mean_r = res.get("mean_reward", 0.0)
    failed_links = res.get("first_failed_link_counts", {})
    achievements = res.get("achievements_unlocked", {})

    print(f"\n▶ Ablation: {name:<26} | Mean Reward: {mean_r:.2f}")
    print(f"  • First Broken Link : {failed_links}")
    if achievements:
        print(f"  • Unlocked Milestones: {achievements}")

print("=" * 80 + "\n")

# 4. Trigger Direct Browser Download
files.download(str(json_path))


 Saved to Drive: /content/drive/MyDrive/learn-a-world/results/phase0_chain_diagnostics.json

 PHASE 0 CHAIN DIAGNOSTICS SUMMARY (3 eps, Seeds: [2050, 2052])

▶ Ablation: strict_no_memory           | Mean Reward: 0.00
  • First Broken Link : {'table_execution': 3}

▶ Ablation: strict_empty_memory        | Mean Reward: 0.00
  • First Broken Link : {'table_execution': 1, 'pickaxe_execution': 1, 'wood_acquisition': 1}

▶ Ablation: strict_preloaded_memory    | Mean Reward: 0.00
  • First Broken Link : {'table_execution': 2, 'wood_acquisition': 1}

▶ Ablation: strict_landmarks_off       | Mean Reward: 0.00
  • First Broken Link : {'table_execution': 1, 'wood_acquisition': 1, 'pickaxe_execution': 1}

▶ Ablation: strict_policy_prior_off    | Mean Reward: 0.00
  • First Broken Link : {'wood_acquisition': 1, 'table_execution': 2}



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Pull the auditor script from GitHub
%cd /content/learn-a-world
!git pull origin main

# 2. Run non-destructive audit across your entire Google Drive
!python3 scripts/audit_drive_data.py \
  --drive_root /content/drive/MyDrive/learn-a-world \
  --output_json results/drive_data_audit_manifest.json \
  --output_md results/drive_data_audit_manifest.md

# 3. Automatically download the markdown manifest to send to Omni
from google.colab import files
files.download("results/drive_data_audit_manifest.md")


/content/learn-a-world
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
Already up to date.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00:1793057744.177720   50763 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/usr/local/lib/python3.13/dist-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_st

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%cd /content/learn-a-world
!git pull origin main

!python3 scripts/run_phase1_5.py \
  --data_root /content/drive/MyDrive/learn-a-world/data \
  --manifest /content/drive/MyDrive/learn-a-world/results/phase1_5_manifest.json \
  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --output_checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_phase1_5.pt \
  --output_json /content/drive/MyDrive/learn-a-world/results/phase1_5_training.json \
  --steps 1000 \
  --batch_size 16 \
  --seq_len 51 \
  --horizons 1 5 10 25 50 \
  --split_seed 17 \
  --device cuda


/content/learn-a-world
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.07 KiB | 99.00 KiB/s, done.
From https://github.com/sahiltolani30/learn-a-world
 * branch            main       -> FETCH_HEAD
   b3b6f90..4547d1f  main       -> origin/main
Updating b3b6f90..4547d1f
Fast-forward
 scripts/run_phase1_5.py     | 10 ++++++++--
 training/sequence_buffer.py | 25 ++++++++++++++++++++++---
 2 files changed, 30 insertions(+), 5 deletions(-)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
I0000 00:00

In [ ]:
%cd /content/learn-a-world

# 1. Fetch latest changes and cleanly reset to origin/main
!git fetch origin
!git reset --hard origin/main

In [ ]:
# Cell: Pull latest Phase 1.5 correctness fixes and run training
%cd /content/learn-a-world
!git pull origin main

!python3 scripts/run_phase1_5.py \
  --data_root /content/drive/MyDrive/learn-a-world/data \
  --manifest /content/drive/MyDrive/learn-a-world/results/phase1_5_manifest.json \
  --checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_t2_final.pt \
  --output_checkpoint /content/drive/MyDrive/learn-a-world/checkpoints/jepa_crafter_v3_phase1_5.pt \
  --output_json /content/drive/MyDrive/learn-a-world/results/phase1_5_training.json \
  --steps 1000 \
  --batch_size 16 \
  --seq_len 51 \
  --labeled_ratio 0.5 \
  --horizons 1 5 10 25 50 \
  --split_seed 17 \
  --device cuda
